In [ ]:
# Import necessary libraries
import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt

# Calculate bubble point for a specific liquid composition
xb_initial = 0.4  # Initial benzene liquid composition
xt_initial = 0.6  # Initial toluene liquid composition

def bubble_point_initial(T):
    P = 760
    Psat_B = 10**(6.90565-(1211.033/(220.79+T)))
    Psat_T = 10**(6.95464-(1344.8/(219.48+T)))
    return Psat_B*xb_initial + Psat_T*xt_initial - P

# Solve for bubble point temperature
Ans_bp = fsolve(bubble_point_initial, 100)
T_bp_initial = Ans_bp[0]

# Calculate vapor composition at bubble point
P = 760
Psat_B = 10**(6.90565-(1211.033/(220.79+T_bp_initial)))
Psat_T = 10**(6.95464-(1344.8/(219.48+T_bp_initial)))
yb = xb_initial * Psat_B / P  # Vapor composition of benzene
yt = xt_initial * Psat_T / P  # Vapor composition of toluene

print(f'Bubble point temperature is {T_bp_initial:.2f}°C')
print(f'Vapor composition in bubble is {yb:.2f} benzene, {yt:.2f} toluene, and sum is {(yb+yt):.0f} as expected')

# Dew point at a Vapor Composition
yben = 0.4
ytou = 0.6

def dew_point(T):
    P = 760
    Psat_B = 10**(6.90565-(1211.033/(220.79+T)))
    Psat_T = 10**(6.95464-(1344.8/(219.48+T)))
    return yben*P/Psat_B + ytou*P/Psat_T - 1

Ans = fsolve(dew_point, 100)
print(f'Dew point is {Ans[0]:.2f}°C')

T_dp = Ans[0]
P = 760
Psat_Ben = 10**(6.90565-(1211.033/(220.79+T_dp)))
Psat_Tou = 10**(6.95464-(1344.8/(219.48+T_dp)))
xben = yben*P/Psat_Ben
xtou = ytou*P/Psat_Tou
print(f'Liquid composition in dew is {xben:.2f} benzene, {xtou:.2f} toluene, and sum is {(xben+xtou):.0f} as expected')

# Calculate bubble point curve (liquid composition vs temperature)
x_benzene = np.linspace(0, 1, 21)
x_toluene = 1 - x_benzene
bubble_T = []
y_benzene_bp = []  # Vapor composition at bubble point
guess = 100

for xb, xt in zip(x_benzene, x_toluene):
    def bubble_point(T):
        P = 760
        Psat_B = 10**(6.90565-(1211.033/(220.79+T)))
        Psat_T = 10**(6.95464-(1344.8/(219.48+T)))
        return Psat_B*xb + Psat_T*xt - P
    
    Ans = fsolve(bubble_point, guess)
    T_bp = float(Ans[0])
    guess = float(Ans[0])  # Update guess for next iteration
    
    # Calculate vapor composition at this bubble point
    Psat_B = 10**(6.90565-(1211.033/(220.79+T_bp)))
    Psat_T = 10**(6.95464-(1344.8/(219.48+T_bp)))
    P = 760
    yb = xb*Psat_B/P  # Vapor composition of benzene
    
    bubble_T.append(T_bp)
    y_benzene_bp.append(yb)

# Calculate dew point curve (vapor composition vs temperature)
y_ben = np.linspace(0, 1, 21)
y_tou = 1 - y_ben
T_dewpoint = []
x_benzene_dp = []  # Liquid composition at dew point

for yben, ytou in zip(y_ben, y_tou):
    def dew_point(T):
        P = 760
        Psat_B = 10**(6.90565-(1211.033/(220.79+T)))
        Psat_T = 10**(6.95464-(1344.8/(219.48+T)))
        return yben*P/Psat_B + ytou*P/Psat_T - 1
    
    Ans = fsolve(dew_point, guess)
    guess = float(Ans[0])  # Update guess for next iteration
    T_dp = float(Ans[0])
    
    # Calculate liquid composition at this dew point
    P = 760
    Psat_Ben = 10**(6.90565-(1211.033/(220.79+T_dp)))
    Psat_Tou = 10**(6.95464-(1344.8/(219.48+T_dp)))
    xben = yben*P/Psat_Ben  # Liquid composition of benzene
    
    T_dewpoint.append(T_dp)
    x_benzene_dp.append(xben)

# Create T-x-y diagram
plt.figure(figsize=(10, 6))
plt.plot(x_benzene, bubble_T, 'b-', label='Bubble point curve (liquid line)', linewidth=2)
plt.plot(y_benzene_bp, bubble_T, 'r-', label='Vapor composition at bubble point', linewidth=2)
plt.plot(x_benzene_dp, T_dewpoint, 'g-', label='Liquid composition at dew point', linewidth=2)
plt.plot(y_ben, T_dewpoint, 'm-', label='Dew point curve (vapor line)', linewidth=2)

plt.xlabel('Mole fraction of benzene')
plt.ylabel('Temperature (°C)')
plt.title('T-x-y Diagram for Benzene-Toluene System')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Printing sample bubble point calculations
print("\nSample Bubble Point Calculations:")
print("Liquid Composition (x_benzene) -> Bubble Point Temperature -> Vapor Composition (y_benzene)")
for i in range(0, len(x_benzene), 5):  # Print every 5th point
    print(f"{x_benzene[i]:.2f} -> {bubble_T[i]:.2f}°C -> {y_benzene_bp[i]:.2f}")